In [1]:
import os

os.environ["MLFLOW_TRACKING_URI"] = "../mlruns"

In [2]:
import pandas as pd

from Doc2VecModel import Doc2VecModel
from DatabaseManager import DatabaseManager

from pymongo import MongoClient

In [3]:
staging_mlflow_model_readme_uri = 'models:/doc2vec_readme@staging'
staging_mlflow_model_others_uri = 'models:/doc2vec_others@staging'

production_mlflow_model_readme_uri = 'models:/doc2vec_readme@production'
production_mlflow_model_others_uri = 'models:/doc2vec_others@production'

staging_model = Doc2VecModel('staging', staging_mlflow_model_readme_uri, staging_mlflow_model_others_uri)
production_model = Doc2VecModel('production', production_mlflow_model_readme_uri, production_mlflow_model_others_uri)

In [4]:
client = MongoClient('localhost', 27017)
database_manager = DatabaseManager(client['github'])

In [5]:
df_vectors = database_manager.get_df_vectors()

In [6]:
print(f'Nombre de vecteurs dans la bd: {len(df_vectors)}')

Nombre de vecteurs dans la bd: 4967


In [7]:
print(df_vectors.iloc[-1]['id'])

377541396


In [8]:
print(df_vectors.iloc[-1]['staging_readme_vector'][:5])

[0.37837305665016174, -0.1239224448800087, -0.1757095605134964, 0.013948597945272923, -0.11249669641256332]


In [9]:
print(df_vectors.iloc[-1]['staging_others_vector'][:5])

[-0.01391507126390934, 0.07616529613733292, -0.0065205832943320274, 0.034448251128196716, 0.054293323308229446]


In [10]:
print(df_vectors.iloc[-1]['production_readme_vector'][:5])

[0.04748052731156349, -0.09738326817750931, -0.29699304699897766, 0.18413619697093964, 0.10551377385854721]


In [11]:
print(df_vectors.iloc[-1]['production_others_vector'][:5])

[-0.02763659879565239, 0.12400985509157181, 0.00595459807664156, 0.050685036927461624, 0.004454581066966057]


In [25]:
df_train_features, df_test_features = database_manager.get_train_test_split_features()

In [12]:
df_train_features

,id,readme_preproc,others_preproc
0,37627792,"[img, align, right, width, 200, src, http, sta...","[document, head, manager, for, react, javascri..."
1,37659549,"[awesome, honeypots, awesome, honeypots, https...","[awesome, list, honeypot, resources, python, a..."
2,37680686,"[awesome, data, engineering, awesome, https, a...","[curated, list, data, engineering, tools, for,..."
3,37688007,"[swubanner, https, raw, githubusercontent, com...","[gem, collection, awesome, crystal, libraries,..."
4,37652385,"[cargo, edit, this, tool, extends, cargo, http...","[utility, for, managing, cargo, dependencies, ..."
...,...,...,...
3995,377421356,"[yet, another, nds, bootstrap, forwarder, yanb...","[yet, another, nds, bootstrap, forwarder, runs..."
3996,377453418,"[功能介绍, 面向各类公共体育场所, 校园对外开放体育设施, 社区健身设施等, 提供场所管理...","[体育场馆预订与扣费小程序, 面向各类公共体育场所, 校园对外开放体育设施, 社区健身设施等..."
3997,377372843,"[img, src, https, raw, githubusercontent, com,...","[sdk, providing, app, protection, and, threat,..."
3998,377609964,[],"[winrar, 简体中文, 商业版, 提取器]"


In [26]:
oriented_df_train_features = df_train_features.to_dict(orient='records')
oriented_df_test_features = df_test_features.to_dict(orient='records')

In [27]:
stg_train_vectors = []
prd_train_vectors = []

stg_test_vectors = []
prd_test_vectors = []

for features in oriented_df_train_features:
    stg_train_vectors.append(staging_model.predict(features))
    prd_train_vectors.append(production_model.predict(features))

for features in oriented_df_test_features:
    stg_test_vectors.append(staging_model.predict(features))
    prd_test_vectors.append(production_model.predict(features))

In [28]:
train_vectors = []
test_vectors = []

for ii, _ in enumerate(stg_train_vectors):
    train_dict = {
        'id': stg_train_vectors[ii]['id'],
        'staging_readme_vector': stg_train_vectors[ii]['staging_readme_vector'],
        'staging_others_vector': stg_train_vectors[ii]['staging_others_vector'],
        'production_readme_vector': prd_train_vectors[ii]['production_readme_vector'],
        'production_others_vector': prd_train_vectors[ii]['production_others_vector'],
    }

    train_vectors.append(train_dict)

for ii, _ in enumerate(stg_test_vectors):
    test_dict = {
        'id': stg_test_vectors[ii]['id'],
        'staging_readme_vector': stg_test_vectors[ii]['staging_readme_vector'],
        'staging_others_vector': stg_test_vectors[ii]['staging_others_vector'],
        'production_readme_vector': prd_test_vectors[ii]['production_readme_vector'],
        'production_others_vector': prd_test_vectors[ii]['production_others_vector'],
    }

    test_vectors.append(test_dict)

In [29]:
cols = [
    'id',
    'staging_readme_vector',
    'staging_others_vector',
    'production_readme_vector',
    'production_others_vector',
]

train_df_vectors_to_insert = pd.DataFrame(
    train_vectors,
    columns=cols
)

test_df_vectors_to_insert = pd.DataFrame(
    test_vectors,
    columns=cols
)

In [30]:
database_manager.insert_repos_vector_for_both_stages(train_df_vectors_to_insert)
database_manager.insert_repos_vector_for_both_stages(test_df_vectors_to_insert)

In [31]:
df_vectors = database_manager.get_df_vectors()

In [32]:
print(f'Nombre de vecteurs dans la bd: {len(df_vectors)}')

Nombre de vecteurs dans la bd: 4967


In [33]:
print(df_vectors.iloc[-1]['id'])

377541396


In [37]:
print(train_df_vectors_to_insert.iloc[-1]['staging_readme_vector'][:5])

[ 0.3764614  -0.12513918 -0.17607617  0.01224301 -0.11092764]


In [34]:
print(df_vectors.iloc[-1]['staging_readme_vector'][:5])

[0.376461386680603, -0.12513917684555054, -0.17607617378234863, 0.01224300917237997, -0.11092764139175415]


In [38]:
print(df_vectors.iloc[-1]['staging_readme_vector'][:5])

[0.376461386680603, -0.12513917684555054, -0.17607617378234863, 0.01224300917237997, -0.11092764139175415]


In [22]:
print(df_vectors.iloc[-1]['staging_others_vector'][:5])

[-0.018234387040138245, 0.07281935960054398, -0.005396431777626276, 0.04610389098525047, 0.060133468359708786]


In [23]:
print(df_vectors.iloc[-1]['production_readme_vector'][:5])

[0.05773910507559776, -0.013423734344542027, -0.3161144256591797, 0.15084154903888702, 0.0787547156214714]


In [24]:
print(df_vectors.iloc[-1]['production_others_vector'][:5])

[-0.0307462140917778, 0.11453843861818314, 0.01185066532343626, 0.05417589843273163, 0.006433820351958275]
